# Dataset Understanding

## Healthcare Readmission Intelligence

Diabetes 130-US Hospitals (1999–2008)

## Business Problem

Hospitals aim to identify patients who are at high risk of being readmitted within
30 days after discharge.

We will only evaluate actionable diabetic encounters. An actionable encounter means:

- The length of stay is between 1 and 14 days (removes 23-hour observation stays and extreme outliers).
- Laboratory tests were performed.
- Medications were administered.

Early identification enables hospitals to

- reduce healthcare costs
- improve patient outcomes
- schedule follow-up care
- allocate healthcare resources efficiently

## Machine Learning Problem

Binary Classification

Target:

readmitted

<30 → 1

\> 30 → 0

NO → 0

## Dataset

Source

UCI Machine Learning Repository

Rows

101,766

Columns

50

In [1]:
import pandas as pd
import numpy as np
import yaml

In [2]:
CONFIG_PATH = '../configs/paths.yaml'

with open(CONFIG_PATH,"r") as file:
    paths = yaml.safe_load(file)

df=pd.read_csv(paths['raw_data']['diabetic_data'])

In [3]:
df.shape

(101766, 50)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

#### Observations

* Dataset contains 50 columns

* Several columns use "?" instead of NaN

* Mix of numerical and categorical variables

* Contains encounter and patient identifiers

* Multiple medication columns

* Three diagnosis columns

* Target is multiclass

In [5]:
df.replace("?",np.nan,inplace=True)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),NaN,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),NaN,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),NaN,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),NaN,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [6]:
missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing>0]

weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64

In [7]:
df['readmitted'].unique()

<StringArray>
['NO', '>30', '<30']
Length: 3, dtype: str

`readmitted` is my target.

We want to find the patients who readmitted in one month - <30

**{'NO':0,  '>30':0,  '<30':1}**

In [8]:
df['readmitted']=df['readmitted'].replace({'NO':0,'>30':0,'<30':1}).astype(int)

In [9]:
identifiers = ['encounter_id', 'patient_nbr']
target = ['readmitted']

numerical = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
    'num_medications', 'number_outpatient', 'number_emergency', 
    'number_inpatient', 'number_diagnoses'
]

diagnosis = ['diag_1', 'diag_2', 'diag_3']

medication = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

categorical = [
    'race', 'gender', 'age', 'payer_code', 'medical_specialty',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 
    'change', 'diabetesMed'
]

print(f"Identifiers/Target ({len(identifiers) + len(target)})")
print(f"Numerical ({len(numerical)}), Categorical ({len(categorical)}), Medication ({len(medication)}), Diagnosis ({len(diagnosis)})")
print(f"Total Columns Mapped: {len(identifiers) + len(target) + len(numerical) + len(categorical) + len(medication) + len(diagnosis)}")

Identifiers/Target (3)
Numerical (8), Categorical (10), Medication (23), Diagnosis (3)
Total Columns Mapped: 47


In [10]:
df["patient_nbr"].duplicated().sum()

np.int64(30248)

## Data Quality Issues

| Issue                 | Impact                             | Planned Solution              |
| --------------------- | ---------------------------------- | ----------------------------- |
| "?" values            | Missing values encoded incorrectly | Convert to NaN                |
| Weight                | 96% missing                        | Drop                          |
| Diagnosis             | Hundreds of ICD-9 codes            | Group into disease categories |
| patient_nbr           | Leakage risk                       | Patient-level split           |
| discharge_disposition | Includes death/hospice (impossible to readmit) | Drop terminal encounters to prevent Class 0 bias |
| A1Cresult | "83% missing implies ""Test Not Ordered"" (strong clinical signal)" | "Impute missing as ""Not Measured"" instead of dropping" |

In [11]:
mapping_data = pd.read_csv(paths['raw_data']['mapping_data'])
mapping_data

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
...,...,...
62,22,Transfer from hospital inpt/same fac reslt in...
63,23,Born inside this hospital
64,24,Born outside this hospital
65,25,Transfer from Ambulatory Surgery Center


In [12]:
df['readmitted'].value_counts(normalize=True)

readmitted
0    0.888401
1    0.111599
Name: proportion, dtype: float64

* The dataset is highly imbalanced only having 11% percentage of the positive class 

### We will

✓ Convert target to binary

✓ Remove identifiers

✓ Group ICD codes

✓ Prevent leakage

✓ Build sklearn Pipeline

✓ Compare baseline vs RF

### Dataset Summary

| Category             |   Count |
| -------------------- | ------: |
| Total Rows           | 101,766 |
| Total Features       |      50 |
| Numerical Features   |      11 |
| Categorical Features |      39 |
| Medication Features  |      24 |
| Diagnosis Features   |       3 |
| Identifier Columns   |       2 |
| Target               |       1 |


Conclusion

We now understand the dataset.

Next notebook:  [02_exploratory_data_analysis.ipynb](02_exploratory_data_analysis.ipynb)